<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:32px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 06 · Probability,<br>Distributions, and Sampling</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 3 November 2026</div>
<div style="font-size:17px;color:#ffffff;border-top:1px solid #466178;padding-top:16px;margin-top:18px;"><strong>Today's challenge:</strong> We can describe the observations we have. But what happens when we see only a sample of a larger group, and how can we reason about uncertainty?</div>
</div>

In **L05**, we described these patterns with numerical statistics. Today we move from **what we observed** to **what could happen when we select or sample observations**.



### What you will be able to do

By the end of this lecture, you should be able to:

- Describe an **uncertain event** and calculate a simple probability from equally likely outcomes or observed proportions.
- Explain how **conditional probability** changes the group we consider, and recognize when events are not independent.
- Explain a **discrete probability distribution**, relate probabilities to frequencies, and interpret an expected value.
- Distinguish **observed histograms**, **continuous density models**, and **cumulative probabilities**.
- Draw reproducible **random samples**, recognize sampling variability, and compare sample means with a fixed reference mean.
- Explain why a **larger random sample** can improve stability but cannot automatically correct **biased selection**.

We will work with one familiar dataset, several small experiments, and four short activities. No previous formal probability course is assumed.

### The same penguins, a new question

Our primary data are the **Palmer Penguins** observations: **344 records** and eight attributes (`species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`, `year`).

**Important boundary:** These 344 records are **not a representative random sample of every penguin in the world**. When we say *select a random penguin record*, we mean choose **one of these 344 rows with equal probability**. Later, we will use recorded body masses as a **fixed reference collection**, not as the population of all penguins.

Some body masses are missing. Whenever we calculate or sample body mass, we will state how many available measurements we use.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
NAVY = "#102A43"
TEAL = "#158B94"
SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_COLORS = {"Adelie": "#158B94", "Chinstrap": "#DB9137", "Gentoo": "#557AB5"}

path = Path("data/penguins.csv")
penguins = pd.read_csv(path)
display(penguins.head())

In [ ]:
print(f"Records in the dataset: {len(penguins)}")
print(f"Body masses recorded: {penguins['body_mass_g'].count()}")
print(f"Body masses missing: {penguins['body_mass_g'].isna().sum()}")
display(penguins['species'].value_counts().reindex(SPECIES_ORDER).rename('Records').to_frame())

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · FROM L05 TO UNCERTAINTY</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">A familiar count, an uncertain next row</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">We know the species counts. Can we know the species of the next random selection?</div>
    </div>

In L05, we calculated means, standard deviations, percentiles, and correlations **for the records we had**. We could describe them precisely.

But suppose we close our eyes and ask the computer to choose **one row uniformly at random**. The row is already in our dataset, yet **which row will be chosen is unknown to us**.

We know there are **152 Adelie**, **68 Chinstrap**, and **124 Gentoo** records. Those counts tell us which outcomes of the selection are more likely.

Think before calculating: Is the next selected species guaranteed to be Adelie just because Adelie has the largest count?

In [ ]:
species_counts = penguins['species'].value_counts().reindex(SPECIES_ORDER)
species_proportions = species_counts / len(penguins)
display(pd.DataFrame({'records': species_counts,
                      'proportion': species_proportions.round(3)}))
print('Total of proportions:', species_proportions.sum())

The proportions describe the **344 records we observed**. They also become exact probabilities for the **specific experiment** of uniformly selecting one of these rows. We must always describe the selection process before interpreting a probability.

The event *"the selected row is Gentoo"* is uncertain. Its probability is not a prediction about every penguin living in nature.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · PROBABILITY AND EVENTS</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Turning uncertain outcomes into numbers</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">What exactly are we counting when we say an outcome is likely?</div>
    </div>

### Start with a small random experiment

Imagine a **fair six-sided die**. A *random experiment* is a process whose result is uncertain before it happens. Its **possible outcomes** are $1,2,3,4,5,6$; together, they form the **sample space** (often written $S$).

An **event** is a question about the outcome. For example, event $A$: *the die shows an even number*. It includes the outcomes $2,4,6$.

If all outcomes are **equally likely**, the probability of event $A$ is

$$
P(A)=\frac{\text{number of outcomes in }A}{\text{total number of possible outcomes}}.
$$

- $P(A)$ means *the probability of event $A$*.
- For a fair die, $P(A)=3/6=0.5$.
- Any probability lies between **0** (impossible) and **1** (certain).

**Question:** What changes if the die is not fair? We can no longer assume that every face has probability $1/6$. We would need an appropriate model or empirical evidence about its behavior.

In [ ]:
die_outcomes = np.arange(1, 7)
even_outcomes = die_outcomes[die_outcomes % 2 == 0]
print('Possible outcomes:', die_outcomes.tolist())
print('Event: an even number:', even_outcomes.tolist())
print('P(even) for a fair die:', len(even_outcomes) / len(die_outcomes))

### The same reasoning for our penguins

Now the possible outcomes are **344 equally likely rows**, not three equally likely species. Since **124 rows are Gentoo**:

$$
P(\text{Gentoo})=\frac{124}{344}\approx 0.3605.
$$

This is about **36.05%**, *for a uniform selection among the observed rows*. It would be incorrect to use $1/3$: the three species do not have equal numbers of rows.

In [ ]:
p_gentoo = (penguins['species'] == 'Gentoo').sum() / len(penguins)
print(f'P(Gentoo) = {p_gentoo:.4f} = {100*p_gentoo:.2f}%')
print('Pandas check:', penguins['species'].value_counts(normalize=True)['Gentoo'])

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(SPECIES_ORDER, species_proportions.loc[SPECIES_ORDER],
       color=[SPECIES_COLORS[s] for s in SPECIES_ORDER])
ax.set(title='Probability of a species when selecting one dataset row uniformly',
       xlabel='Penguin species', ylabel='Selection probability', ylim=(0, 0.52))
for i, p in enumerate(species_proportions.loc[SPECIES_ORDER]):
    ax.text(i, p + 0.016, f'{p:.1%}', ha='center')
plt.show()

### "Not", "or", and "and" are different events

Suppose $G$ means *the selected record is Gentoo*.

- **Not Gentoo** is the *complement* of $G$: $P(\text{not }G)=1-P(G)=220/344$.
- **Adelie or Gentoo:** A selected row has one species label, so these are **mutually exclusive** alternatives. Add the two probabilities: $(152+124)/344=276/344$.
- **Adelie and Gentoo:** For one row with exactly one species label, this is impossible, so the probability is **0**.

But not all "and" events are impossible. *Gentoo and Biscoe Island* can occur together: species and island describe different attributes of the **same selected record**. We will investigate this next.

The simple addition rule above applies to **mutually exclusive** events. If events overlap, adding their probabilities would count some outcomes twice.

In [ ]:
p_not_gentoo = 1 - p_gentoo
p_adelie_or_gentoo = species_proportions['Adelie'] + species_proportions['Gentoo']
p_gentoo_and_biscoe = ((penguins['species'] == 'Gentoo') &
                       (penguins['island'] == 'Biscoe')).mean()
print(f'P(not Gentoo): {p_not_gentoo:.4f}')
print(f'P(Adelie or Gentoo): {p_adelie_or_gentoo:.4f}')
print(f'P(Gentoo and Biscoe): {p_gentoo_and_biscoe:.4f}')

<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 1 — From counts to probabilities</strong></div>

**Your task:** Use the actual 344 rows.

1. Calculate the probability of each species when a row is selected uniformly.
2. Find the probability of selecting a record that is **not Chinstap**.
3. Verify that the three species probabilities sum to **1**.

*Try the question and edit the next code cell before expanding the answer.*

In [ ]:
# Activity 1: Complete the expressions below.
# Hint: .value_counts(normalize=True) and .sum() may help.
activity1_probs = penguins['species'].value_counts(normalize=True)
# TODO: display probabilities in species order; calculate not-Gentoo; check the sum.

<details><summary><strong>Suggested answer — click to expand after trying</strong></summary>

**Suggested code**

```python
activity1_probs = penguins['species'].value_counts(normalize=True).reindex(SPECIES_ORDER)
display(activity1_probs)
print('Not Chinstap:', 1 - activity1_probs['Chinstap'])
print('Total:', activity1_probs.sum())
```

**Interpretation:** Adelie is $152/344\approx44.19\%$, Chinstrap is $68/344\approx19.77\%$, and Gentoo is $124/344\approx36.05\%$. Not Chinstap is $276/344\approx63.95\%$. They sum to 1 because the three labels cover all 344 rows without overlap. None of these quantities establishes the species proportions of all wild penguins.

</details>

### Theoretical probability and observed frequency are related, not identical

- A fair coin has a **theoretical** probability of $0.5$ for heads. That probability comes from our *fair-coin model*.

- If we toss it 10 times, we might observe 4 heads or 7 heads. The **empirical relative frequency** is

$$
\text{relative frequency of heads}=\frac{\text{number of observed heads}}{\text{number of tosses}}.
$$

- It can be different from $0.5$ in any finite set of tosses. With more independent tosses from the same fair-coin model, it often becomes **closer** to $0.5$; no fixed number of tosses guarantees a perfect match.

- In the penguin example, our row-selection probabilities come from the **observed counts plus the chosen uniform-row selection rule**, not from an assumed natural species distribution.

In [ ]:
rng_coin = np.random.default_rng(10)
for n_tosses in [10, 100, 10_000]:
    # 1 represents heads, 0 represents tails for a fair-coin model.
    tosses = rng_coin.integers(0, 2, size=n_tosses)
    print(f'{n_tosses:>6} tosses: {tosses.mean():.3f} observed heads; model P(heads) = 0.500')

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · CONDITIONAL PROBABILITY</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">When new information changes the reference group</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Does knowing the island change the chance of selecting Gentoo?</div>
    </div>

Selecting a random row, we found $P(\text{Gentoo})\approx 0.3605$. Now imagine we are told **the selected penguin was recorded on Biscoe Island**.

We are no longer choosing among all 344 records. Our reference group is only the records from **Biscoe**. The island information changes the **denominator**. Before calculating, predict: will Gentoo become more or less likely?

In [ ]:
island_species = pd.crosstab(penguins['island'], penguins['species'])
display(island_species.reindex(columns=SPECIES_ORDER))

n_biscoe = (penguins['island'] == 'Biscoe').sum()
n_biscoe_gentoo = ((penguins['island'] == 'Biscoe') &
                   (penguins['species'] == 'Gentoo')).sum()
print('Biscoe records:', n_biscoe)
print('Gentoo recorded on Biscoe:', n_biscoe_gentoo)
print('P(Gentoo | Biscoe):', n_biscoe_gentoo / n_biscoe)

### Name the idea: conditional probability

A **conditional probability** is the probability of event $A$ **given** that condition $B$ is true. We write $P(A\mid B)$, read as *probability of A given B*.

For our question:

- $A$: the selected record is Gentoo.
- $B$: the selected record is from Biscoe Island.
- $A\cap B$: both statements are true for that same record (the symbol $\cap$ means *and*).

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)},\qquad P(B)>0.
$$

**Why does this work?** If the row is known to be Biscoe, we consider only Biscoe rows. Among **168 Biscoe records**, **124 are Gentoo**:

$$
P(\text{Gentoo}\mid\text{Biscoe})=\frac{124}{168}\approx0.7381.
$$

This is about **73.81%**, much higher than 36.05% without island information.

In [ ]:
p_biscoe = n_biscoe / len(penguins)
p_both = n_biscoe_gentoo / len(penguins)
p_gentoo_given_biscoe = p_both / p_biscoe
print(f'P(Biscoe) = {p_biscoe:.4f}')
print(f'P(Gentoo and Biscoe) = {p_both:.4f}')
print(f'P(Gentoo | Biscoe) = {p_gentoo_given_biscoe:.4f}')
assert np.isclose(p_gentoo_given_biscoe, 124 / 168)

### A comparison across islands

We can repeat that calculation for every species **within each island**. Here, each **row** of the table adds to 1 because we are conditioning on one island at a time.

In [ ]:
by_island = pd.crosstab(penguins['island'], penguins['species'], normalize='index')
by_island = by_island.reindex(columns=SPECIES_ORDER)
display(by_island.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
by_island.plot(kind='bar', stacked=True, ax=ax,
               color=[SPECIES_COLORS[s] for s in SPECIES_ORDER], width=0.65)
ax.set(title='Species probabilities conditional on island (record selection)',
       xlabel='Recorded island', ylabel='Conditional probability', ylim=(0, 1.02))
ax.legend(title='Species', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### A direction matters: $P(A\mid B)$ is not $P(B\mid A)$

All Gentoo records in this dataset are from Biscoe. Therefore:

$$
P(\text{Biscoe}\mid\text{Gentoo})=\frac{124}{124}=1.
$$

However, $P(\text{Gentoo}\mid\text{Biscoe})=124/168\approx0.7381$, because Biscoe also has Adelie records.

**Intuitive independence:** If knowing $B$ does not change the probability of $A$, we call the events **independent** (provided the conditioning event has positive probability). Here, island and species are **not independent** under uniform record selection: knowing the island changes species probabilities. This is an association in our recorded rows, not evidence that an island causes a species.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · RANDOM VARIABLES AND DISTRIBUTIONS</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">From one event to all possible outcomes</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Can we describe a random experiment using a table of probabilities?</div>
    </div>

### A random variable gives a numerical result

Imagine tossing **two fair coins**. The four equally likely ordered outcomes are **HH, HT, TH, TT**. We define a **random variable** $X$:

> $X=$ **the number of heads in the two tosses**.

The outcome is a sequence of coin sides, but $X$ turns that sequence into a **number**: 0, 1, or 2. Different outcomes can give the same value (both HT and TH give $X=1$).

| Ordered outcome | HH | HT | TH | TT |
|:--|:--:|:--:|:--:|:--:|
| Number of heads, $X$ | 2 | 1 | 1 | 0 |

Because our two-coin model treats the four sequences as equally likely, we obtain:

| $x$ (heads) | 0 | 1 | 2 |
|:--|:--:|:--:|:--:|
| $P(X=x)$ | $1/4$ | $2/4$ | $1/4$ |

This is a **discrete probability distribution**: a list of possible numerical values and their probabilities. The function that assigns $P(X=x)$ is called the **probability mass function (PMF)**.

Its probabilities must be nonnegative and add to **1**.

### Connect the expected value to the mean

In L05, the mean gave every **observed value** an equal weight. A distribution lets us calculate an analogous theoretical center: the **expected value**.

$$
E[X]=\sum_x x\,P(X=x).
$$

Read it as: *multiply each possible numerical value $x$ by its probability, then add the results*.

For our two tosses:

$$
E[X]=0\times\frac14+1\times\frac12+2\times\frac14=1.
$$

We **expect 1 head on average over many two-toss experiments**, not exactly one head in each experiment. The expected value is a property of the probability model; the **sample mean** calculated from a limited number of experiments may differ.

### Compare a model with repeated observations

The theoretical PMF **does not change** when we run more simulations. What changes is how accurately our **observed relative frequencies** reflect it. We can repeat the same two-toss experiment, counting the number of heads each time.


### A small connection back to penguins

A row's species is a **category**, not a numerical random variable. However, we can define a numerical indicator: $I=1$ if the randomly selected row is Gentoo and $I=0$ otherwise. Under uniform row selection, $P(I=1)=124/344$.

This gives a two-value discrete distribution. It also shows how a probability can be represented by the **mean of a 0/1 indicator**, connecting back to Pandas boolean averages.

In [ ]:
gentoo_indicator = (penguins['species'] == 'Gentoo').astype(int)
print('Observed mean of Gentoo indicator:', gentoo_indicator.mean())
print('Selection probability of Gentoo:', p_gentoo)
assert np.isclose(gentoo_indicator.mean(), p_gentoo)

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · CONTINUOUS VALUES</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">From counts to measurement distributions</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">What changes when the result is a measurement such as body mass?</div>
    </div>

### Two kinds of numerical uncertainty

With two coin tosses, $X$ has only three values: 0, 1, 2. This is **discrete**.

- A body mass is a measurement. Conceptually, it can vary over a range of values, so it is often useful to represent it using a **continuous probability model**. Recorded masses have finite precision (and our file contains a finite number of values); the **continuous model** is an idealization, not a claim that recorded measurements have infinitely many decimal places.
- Remember the L04 histogram: it counts *recorded measurements in intervals*. A continuous **probability density curve** instead describes a *model* of how probability is spread across intervals.

The crucial idea is:

- For a **discrete PMF**, a bar height is the probability of one listed value.
- For a **continuous density**, the *area under the curve between two values* is the probability of an interval. **Density height is not itself a probability**.
- In an ideal continuous model, one exact value has probability zero; an interval has positive probability.

We will not need integration or a density formula to understand this.

### Two familiar models: uniform and normal

A **continuous uniform distribution** assigns equal probability **density** across a range. For example, a number randomly generated between 0 and 1 is equally likely to fall in two subintervals of the same width.

A **normal distribution** has a bell-shaped density, controlled by a **mean** (center) and a **standard deviation** (spread). It is useful for some measured phenomena, but **not every dataset is normal**.

Both plots below show **simulated values from stated mathematical models**, not observed penguin data. We use fixed seeds so everyone sees the same draws.

In [ ]:
rng_models = np.random.default_rng(606)
uniform_draws = rng_models.uniform(0, 1, size=5000)
normal_draws = rng_models.normal(loc=0, scale=1, size=5000)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.7))
axes[0].hist(uniform_draws, bins=26, density=True, color=TEAL, edgecolor='white')
axes[0].set(title='Uniform(0, 1): simulated values', xlabel='Generated value',
            ylabel='Empirical density')
axes[1].hist(normal_draws, bins=35, density=True, color='#557AB5', edgecolor='white')
axes[1].set(title='Normal(0, 1): simulated values', xlabel='Generated value',
            ylabel='Empirical density')
plt.tight_layout()
plt.show()

Consider the **uniform model** on $[0,1)$. Its probability of producing a number from $0.2$ up to (but not including) $0.5$ is **0.3**, the width of that interval in this particular model. We can check whether simulations give approximately the same fraction.

For a continuous model, $P(Y=0.5)=0$ for one exact value. This does **not** say intervals around 0.5 are impossible, and it does not imply that a *rounded recorded measurement* cannot be equal to 0.5.

In [ ]:
theoretical_interval_prob = 0.5 - 0.2
observed_interval_fraction = ((uniform_draws >= 0.2) & (uniform_draws < 0.5)).mean()
print(f'Uniform-model P(0.2 <= Y < 0.5): {theoretical_interval_prob:.3f}')
print(f'Fraction from 5,000 simulated draws: {observed_interval_fraction:.3f}')

### Return to real body-mass observations

Our 344-row dataset has **342 recorded body masses** and **2 missing masses**. Here, the histogram below describes those **342 measurements**, not a theoretical probability law. We may see a complicated pooled shape partly because we combined different species.

Do not assume that the pooled distribution follows the normal model simply because we know its mean and standard deviation.

In [ ]:
mass_by_species = penguins.dropna(subset=['body_mass_g'])

fig, ax = plt.subplots(figsize=(9, 4))
for species in SPECIES_ORDER:
    values = mass_by_species.loc[mass_by_species['species'] == species, 'body_mass_g']
    ax.hist(values, bins=np.arange(2500, 6650, 250), alpha=0.57,
            color=SPECIES_COLORS[species], label=f'{species} (n={len(values)})')
ax.set(title='Observed body masses by species (342 recorded measurements)',
       xlabel='Recorded body mass (g)', ylabel='Number of recorded penguins')
ax.legend()
plt.show()

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · SAMPLING AND VARIABILITY</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">What can a smaller set of observations tell us?</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">If we select new random samples, will we obtain the same mean every time?</div>
    </div>

### Population, sample, parameter, statistic

In a real study, the **population** is the full group we want to understand. A **sample** is the subset we actually observe. A numerical description of a population, such as its mean $\mu$, is a **parameter**. A numerical description calculated from a sample, such as its mean $\bar{x}$, is a **statistic**.

In this lecture's simulation, we define a narrower, *known* population for learning purposes: the **342 recorded penguin body masses**, treated as a **fixed reference collection**. Its complete mean is therefore known; we can repeatedly take smaller samples and compare their means against it.

This fixed collection is **not the population of all real penguins**. A random sample from these 342 values teaches us about selection variability *within this collection*. It does not validate how the original penguin records were gathered.

- Reference mean: $\mu_{\mathrm{ref}}=\frac1N\sum_{i=1}^{N}x_i$, with $N=342$ available measurements.
- Sample mean: $\bar{x}=\frac1n\sum_{i=1}^{n}x_i$, with a sample of size $n$.

The symbol $x_i$ means a recorded body-mass value (in grams). A different random sample can have a different $\bar{x}$ even when the reference collection never changes.

In [ ]:
masses = penguins['body_mass_g'].dropna()
reference_values = masses.to_numpy()  # exactly 342 available masses
N = len(reference_values)
reference_mean = reference_values.mean()
reference_sd = reference_values.std(ddof=0)
print(f'Fixed reference N = {N} measured masses')
print(f'Reference mean = {reference_mean:,.2f} g')
print(f'Reference SD across individual masses = {reference_sd:,.2f} g')

### Draw a random sample (without replacement)

**Simple random sampling without replacement:** every subset of a specified size is equally likely. Once a row is selected **within that sample**, we do not select that same row again.

**With replacement:** the same row can be selected more than once during one sample; each draw can independently choose from the full collection. Sampling with replacement is useful in other settings, but our main demonstrations will use **without replacement** to mimic selecting distinct records.

We use `np.random.default_rng(seed)` to get **reproducible sequences**. A seed repeats the computer experiment; it does not make a real-world selection perfectly representative.

In [ ]:
rng_sample = np.random.default_rng(612)
sample_12a = rng_sample.choice(reference_values, size=12, replace=False)
sample_12b = rng_sample.choice(reference_values, size=12, replace=False)
print(f'Full reference mean: {reference_mean:,.1f} g')
print(f'Sample A (n=12) mean: {sample_12a.mean():,.1f} g')
print(f'Sample B (n=12) mean: {sample_12b.mean():,.1f} g')
print('Same reference collection; two different selected samples.')

**A subtle point:** Some penguins have the same *recorded mass*. Two identical mass numbers do **not necessarily mean the same record was selected twice**. `replace=False` prevents repeating a *row position* during a sample; equal values may still occur because several different rows have equal mass.

For clarity, this tiny artificial example uses unique labels to illustrate **replacement**, not the penguin data:

In [ ]:
rng_replacement = np.random.default_rng(77)
labels = np.array(['A', 'B', 'C'])
print('Without replacement:', rng_replacement.choice(labels, size=3, replace=False))
print('With replacement:', rng_replacement.choice(labels, size=6, replace=True))

### Repeat the sampling experiment

Instead of looking at just two samples, draw **800 independent random samples** of the same size from our fixed collection. We do **not replace records inside each sample**, but we start again from the full 342 masses for each new sample.

For each sample, calculate one **sample mean**. We will compare $n=10$ with $n=80$.

Notice the difference between these two kinds of variability:

1. **Individual-body-mass variability:** recorded penguins differ in body mass.
2. **Sample-mean variability:** estimates from repeated random samples differ because different records were selected.

The two are related, but the plotted objects below are **means of samples**, not the 342 original measurements.

In [ ]:
rng_repeated = np.random.default_rng(613)
repetitions = 800
means_n10 = np.array([rng_repeated.choice(reference_values, size=10, replace=False).mean()
                     for _ in range(repetitions)])
means_n80 = np.array([rng_repeated.choice(reference_values, size=80, replace=False).mean()
                     for _ in range(repetitions)])
print(f'Individual body-mass SD (342 records): {reference_sd:.1f} g')
print(f'SD of {repetitions} sample means (n=10): {means_n10.std(ddof=1):.1f} g')
print(f'SD of {repetitions} sample means (n=80): {means_n80.std(ddof=1):.1f} g')

bins = np.linspace(min(means_n10.min(), means_n80.min()) - 50,
                   max(means_n10.max(), means_n80.max()) + 50, 25)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, sharey=True)
for ax, values, n, color in [(axes[0], means_n10, 10, TEAL),
                              (axes[1], means_n80, 80, '#557AB5')]:
    ax.hist(values, bins=bins, color=color, edgecolor='white')
    ax.axvline(reference_mean, color='#DB9137', linestyle='--', linewidth=2,
               label=f'Reference mean = {reference_mean:,.0f} g')
    ax.set(title=f'{repetitions} random samples, n={n}',
           xlabel='Sample mean body mass (g)', ylabel='Number of samples')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Interpretation:** In this reproducible simulation, the means of samples of 80 are more concentrated near the **fixed reference mean** than the means of samples of 10. That is an illustration of **sampling variability**. The sample means do not all equal the reference mean, and the variation is different from the SD of individual penguins.

It is not a guarantee that **every particular** larger sample has a mean closer to the reference mean than **every particular** smaller sample. Sampling is still random. Nor does this demonstrate anything about representativeness of penguins outside the original records.

### Can a larger sample fix biased selection?

Imagine that we want the **mean of all 342 recorded masses**, but our method selects **only Gentoo** rows. Even if we select many Gentoo records, we are sampling from a group with a different mean.

**Sampling bias** is a systematic problem with the way observations are selected. It is different from the random variation between samples. Increasing the number of observations from the **wrong group** does not remove that mismatch.

This would **not** be biased if our research question were specifically about Gentoo penguins. Whether a selection is biased depends on the **target group and question**.

In [ ]:
gentoo_masses = penguins.loc[penguins['species'] == 'Gentoo', 'body_mass_g'].dropna().to_numpy()
rng_biased = np.random.default_rng(614)
gentoo_only_100 = rng_biased.choice(gentoo_masses, size=100, replace=False)
print(f'All recorded masses, N={N}: mean {reference_mean:,.1f} g')
print(f'Gentoo-only masses, N={len(gentoo_masses)}: mean {gentoo_masses.mean():,.1f} g')
print(f'Random 100 Gentoo masses: mean {gentoo_only_100.mean():,.1f} g')
print('The 100-row sample is from a different target group.')

That matters for **Machine Learning**: a model learns from the examples it sees. If the data exclude important groups or conditions, more examples from the same narrow selection may not help the model work well elsewhere.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · BRING IT TOGETHER</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Next challenge</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">We can reason about uncertainty. What if our underlying observations are imperfect?</div>
    </div>

### Where we go next — L07: Data Quality

Today we saw that **what we select** influences the conclusions we reach. But even after selecting records, their **quality** matters.

> **What if the observations are missing, incorrect, inconsistent, or duplicated?**

In **L07 — Data Quality: Cleaning and Understanding Imperfect Data**, we will examine these problems directly. We did **not** fill in missing body masses today; we identified them and stated when calculations used only valid values.

# Assignment 03 — Preparing Data for Machine Learning: Finding a Representative Split

## The challenge

Throughout the previous lectures, we learned how to explore datasets, visualize patterns, calculate descriptive statistics, understand probability, and perform random sampling.

Now, imagine that we are preparing the **Palmer Penguins dataset** for a future Machine Learning model that will predict penguin species from physical measurements.

Before training the model, we need to divide the dataset into three parts:

| Dataset | Proportion | Purpose |
|---|---:|---|
| Training | 70% | Used to learn patterns from data |
| Validation | 20% | Used to help select and improve a model |
| Test | 10% | Kept separate for evaluating the final model |

But there is an important question:

**If we randomly divide the data, will each part represent the characteristics of the original dataset?**

For example, a random split might contain too many Gentoo penguins in the training set and too few in the test set. The same problem may occur with the islands or with the distribution of body mass.

This could make the resulting evaluation less reliable.

**Your mission:** Investigate 1,000 different random splits, identify the three that best preserve the original dataset's species and island proportions, and select one final split using descriptive statistics to support your decision.

---

## Part 1 — Understand the original dataset

Load `data/penguins.csv` and investigate the complete dataset.

1. Report the number of observations and attributes.
2. Calculate the number and percentage of penguins belonging to each **species** and **island**.
3. Calculate the **mean, median, and standard deviation of body mass**, separately for each species and each island.
4. Create an appropriate visualization showing the distribution of species and islands.

These results are your **reference statistics**. You will use them to evaluate the quality of your candidate splits.



## Part 2 — Generate 50 random splits

Use random sampling to divide the dataset into training (70%), validation (20%), and test (10%) sets.

Repeat the procedure **50 times**, using a different random selection for each candidate.

You may use Pandas or NumPy. Use a fixed random seed, or reproducibly generated seeds, so that your experiment can be repeated.

For every candidate:

- All original records must be assigned to exactly one split.
- The three splits must not overlap.
- The split sizes must be as close as possible to 70%, 20%, and 10%.
- Calculate the proportions of each species and each island in all three splits.

Suggested code to sample the data:

``` Python
# Stratified random split: keep species represented
train_strat = penguins.groupby("species").sample(
    frac=0.70
)
remaining = penguins.drop(train_strat.index)

val_strat = remaining.groupby("species").sample(
    frac=0.50
)
test_strat = remaining.drop(val_strat.index)
.
.
.
```


## Part 3 — Find the three closest distributions

We want to identify the candidate splits that most closely preserve the characteristics of the original dataset.

For each candidate, compare the species and island proportions in the training, validation, and test sets with those in the complete dataset.

For example, if Gentoo penguins represent 36% of the complete dataset but only 30% of a particular split, the absolute difference is **6 percentage points**.

Define a simple **distribution difference score**:

1. Calculate the absolute percentage-point difference for every species and island in each of the three splits.
2. Compute the average of these differences to obtain one score for the candidate.
3. Repeat for all 50 candidates.
4. Rank the candidates from the smallest score (closest) to the largest score.

A smaller score means the candidate better preserves the original **species and island proportions**, according to this measure.

Identify the **three candidates with the lowest scores**.

Present their scores and compare their species and island proportions using a table or visualization.



## Part 4 — Save your selected dataset

For the best out of the three closes distributions, add a new column named `split` to the **original dataset**.

Every record must contain exactly one of the following labels:

- `train`
- `val`
- `test`

Save your result as:

`penguins_with_splits.csv`

Verify that:

- All 344 original records are preserved.
- All original attributes remain unchanged.
- Every record has a valid split label.
- Approximately 70% of records are labeled `train`, 20% `val`, and 10% `test`.
- The species and island distributions are close to those of the original dataset.

Your final CSV will be ready for use in a future Machine Learning lecture.



## Part 5 — Write a markdown Data Science report in your notebook

In your notebook you should communicating your investigation and findings.
You should answer these five questions:

1. **What did the original dataset reveal?** Summarize its important characteristics and descriptive statistics.
2. **How did random sampling affect the distributions?** Explain what you observed across the 50 candidate splits.
3. **Which three candidates were closest to the original dataset?** Present their distribution difference scores and briefly compare them.
4. **Why did you select your final candidate?** Support your choice with evidence from species proportions, island proportions, and body-mass statistics.
5. **What are the limitations of your approach?** Discuss whether selecting the closest split guarantees a representative or reliable future Machine Learning evaluation.

Use one compact table or figure where it helps communicate your findings. Focus on interpreting your results rather than listing all the calculations.




## Submission

Submit the following two files:

| File | Content |
|---|---|
| `Assignment03.ipynb` | Complete executable Python analysis, including the 50 random splits, comparisons, ranking, visualizations, and selected candidates, with markdown explinations and questions answers |
| `penguins_with_splits.csv` | Original dataset with the new `split` column |

Your notebook should run sequentially and reproduce your final selected split.

**DD: 10th of Nov 2026 before 18:00.**

### Sources, acknowledgements, and further reading

- **Palmer Penguins dataset:** Horst, A. M., Hill, A. P., & Gorman, K. B. (2020), *palmerpenguins: Palmer Archipelago (Antarctica) penguin data* ([documentation](https://allisonhorst.github.io/palmerpenguins/), [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)); original ecological observations: Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014), *PLoS ONE*. The offline `penguins.csv` is the standard 344-row, eight-column public-domain (CC0) version.
- **Teaching inspirations:** Selected concepts were adapted from the teaching material of **Prof. Antonio Liotta**.
- **NumPy:** [`default_rng`](https://numpy.org/doc/stable/reference/random/generator.html), [`Generator.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html).
- **Pandas:** [`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html), [`crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html).
- **Matplotlib:** [`hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.hist.html), [`step`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.step.html).

The penguin data and demonstrations are bundled for **offline execution**. Random-number simulations show **one reproducible example**, not universal numerical outcomes.